# Feature-Rich FUSE — Full Benchmark Runner

This notebook runs all four feature-rich FUSE strategies across multiple datasets, seeds, and mask fractions, and saves results to `./feature rich fuse/benchmark/`.

**Strategies:**
- `s1` — Feature-Initialized Embeddings
- `s2` — Feature-Aware Attention in Random Walks  
- `s3` — Feature Reconstruction Gradient
- `s4` — Feature-Augmented Adjacency

**How to use:** Configure the `CONFIG` cell below, then run all cells.

> Results are saved per-strategy and aggregated in `./feature rich fuse/benchmark/`.

## 0. Imports

In [3]:
import os, time, random, pickle
import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.sparse import csr_matrix, lil_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
from torch_geometric.datasets import Planetoid, WikiCS, Amazon
from torch_geometric.data import Data as PyGData
from spektral.datasets import Cora

import tensorflow as tf
from spektral.layers import GCNConv, GATConv, GraphSageConv

import logging
logging.getLogger('gensim').setLevel(logging.ERROR)

SAVE_ROOT = os.path.join('./feature rich fuse', 'benchmark')
os.makedirs(SAVE_ROOT, exist_ok=True)
print(f'Results root: {os.path.abspath(SAVE_ROOT)}')

Results root: C:\Users\user\Downloads\FUSE\modularity_intergrated_version\modularity_intergrated_version\feature rich fuse\benchmark


## 1. Benchmark Configuration

**Edit this cell to control what gets run.**

In [5]:
CONFIG = dict(
    # ── Datasets ──────────────────────────────────────────────────────────────
    # Options: 'cora', 'citeseer', 'pubmed', 'wikics', 'photo'
    datasets     = ['cora', 'citeseer', 'pubmed', 'wikics', 'photo'],

    # ── Experimental conditions ───────────────────────────────────────────────
    seeds        = [42],
    mask_fracs   = [0.7, 0.3],   # fraction of labels KNOWN

    # ── Which strategies to run ───────────────────────────────────────────────
    # Options: 's1', 's2', 's3', 's4'
    strategies   = ['s1', 's2', 's3', 's4'],

    # ── Classifiers ───────────────────────────────────────────────────────────
    classifiers  = ['gcn', 'gat', 'graphsage'],

    # ── Embedding hyperparameters ─────────────────────────────────────────────
    emb_dim      = 150,
    iterations   = 200,          # FUSE gradient steps
    clf_epochs   = 200,          # classifier training epochs

    # ── Strategy-specific hyperparameters ────────────────────────────────────
    # S2: blend weight (0=feature only, 1=structural only)
    s2_alpha     = 0.5,
    # S3: feature reconstruction loss weight
    s3_lambda_feature = 0.5,
    # S4: kNN graph blend weight and neighborhood size
    s4_beta      = 0.3,
    s4_knn_k     = 10,

    # ── FUSE walk parameters ─────────────────────────────────────────────────
    num_walks           = 10,
    walk_length         = 5,
    walk_length_labelled= 3,

    # ── Misc ─────────────────────────────────────────────────────────────────
    data_root    = '.',
    verbose      = False,
)

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:25s}: {v}')

Configuration:
  datasets                 : ['cora', 'citeseer', 'pubmed', 'wikics', 'photo']
  seeds                    : [42]
  mask_fracs               : [0.7, 0.3]
  strategies               : ['s1', 's2', 's3', 's4']
  classifiers              : ['gcn', 'gat', 'graphsage']
  emb_dim                  : 150
  iterations               : 200
  clf_epochs               : 200
  s2_alpha                 : 0.5
  s3_lambda_feature        : 0.5
  s4_beta                  : 0.3
  s4_knn_k                 : 10
  num_walks                : 10
  walk_length              : 5
  walk_length_labelled     : 3
  data_root                : .
  verbose                  : False


## 2. Dataset Loader & Masking

In [7]:
def load_dataset(dataset_name, root='.'):
    name = dataset_name.lower()
    if name == 'cora':
        data = Cora()
        graph = data.graphs[0]
        x = graph.x
        a = graph.a.tocsr() if sp.issparse(graph.a) else csr_matrix(graph.a)
        y_onehot = graph.y
        labels = np.argmax(y_onehot, axis=1)
    elif name in ('citeseer', 'pubmed'):
        data = Planetoid(root=root, name=dataset_name.capitalize())
        d = data[0]
        x = d.x.numpy(); edge_index = d.edge_index.numpy(); labels = d.y.numpy()
        num_nodes = x.shape[0]
        a = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            s, t = edge_index[:, i]; a[s, t] = 1; a[t, s] = 1
        a = a.tocsr(); y_onehot = np.eye(labels.max() + 1)[labels]
    elif name == 'wikics':
        data = WikiCS(root=root)
        d = data[0]
        x = d.x.numpy(); edge_index = d.edge_index.numpy(); labels = d.y.numpy()
        num_nodes = x.shape[0]
        a = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            s, t = edge_index[:, i]; a[s, t] = 1; a[t, s] = 1
        a = a.tocsr(); y_onehot = np.eye(labels.max() + 1)[labels]
    elif name in ('photo', 'amazon-photo', 'amazon_photos'):
        data = Amazon(root=root, name='photo')
        d = data[0]
        x = d.x.numpy(); edge_index = d.edge_index.numpy(); labels = d.y.numpy()
        num_nodes = x.shape[0]
        a = lil_matrix((num_nodes, num_nodes), dtype=np.float32)
        for i in range(edge_index.shape[1]):
            s, t = edge_index[:, i]; a[s, t] = 1; a[t, s] = 1
        a = a.tocsr(); y_onehot = np.eye(labels.max() + 1)[labels]
    else:
        raise ValueError(f'Unknown dataset: {dataset_name}')

    row, col = a.nonzero()
    pyg = PyGData(x=torch.tensor(x, dtype=torch.float),
                  edge_index=torch.tensor(np.vstack([row, col]), dtype=torch.long),
                  y=torch.tensor(labels, dtype=torch.long))
    return dict(x=np.array(x, dtype=float), a=a.tocsr(),
                y=np.array(y_onehot, dtype=float),
                labels=np.array(labels, dtype=int),
                G=nx.from_scipy_sparse_array(a), pyg_data=pyg)


def create_label_mask(labels, mask_frac=0.7, seed=None):
    n = len(labels)
    rng = np.random.RandomState(seed) if seed is not None else np.random
    k = int(round(n * (1 - mask_frac)))
    hidden = rng.choice(np.arange(n), size=k, replace=False)
    masked = np.array(labels, dtype=int)
    masked[hidden] = -1
    return masked, masked != -1, hidden

print('Data utilities defined.')

Data utilities defined.


## 3. Shared FUSE Primitives

In [9]:
def perform_labeled_random_walks(G, label_mask, labels, num_walks=10,
                                  walk_length=5, walk_length_labelled=3):
    walks = {node: [] for node in G.nodes()}
    for node in G.nodes():
        for _ in range(num_walks):
            walk = [node]; labeled_count = 0
            for _ in range(walk_length - 1):
                cur = walk[-1]; nbrs = list(G.neighbors(cur))
                if not nbrs: break
                labeled_nbrs = [nb for nb in nbrs if label_mask[nb]]
                if labeled_nbrs and labeled_count < walk_length_labelled:
                    nxt = random.choice(labeled_nbrs); labeled_count += 1
                else:
                    nxt = random.choice(nbrs)
                walk.append(nxt)
            walks[node].extend([nb for nb in walk if label_mask[nb]])
    return walks


def attn_base(S, labeled_nodes):
    weights = {}
    for node, labeled in labeled_nodes.items():
        if labeled:
            sims = {nb: float(np.dot(S[node], S[nb])) for nb in labeled}
            exp_s = {nb: np.exp(v) for nb, v in sims.items()}
            tot = sum(exp_s.values()) or 1.0
            weights[node] = {nb: exp_s[nb] / tot for nb in labeled}
    return weights


def attn_feature_aware(S, labeled_nodes, X, alpha=0.5):
    X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
    weights = {}
    for node, labeled in labeled_nodes.items():
        if labeled:
            sims = {}
            for nb in labeled:
                s_sim = float(np.dot(S[node], S[nb]))
                f_sim = float(np.dot(X_norm[node], X_norm[nb]))
                sims[nb] = alpha * s_sim + (1.0 - alpha) * f_sim
            exp_s = {nb: np.exp(v) for nb, v in sims.items()}
            tot = sum(exp_s.values()) or 1.0
            weights[node] = {nb: exp_s[nb] / tot for nb in labeled}
    return weights


def mod_grad(A, S, degrees, m):
    return (1 / (2 * m)) * (A @ S - (degrees[:, None] / (2 * m)) * S.sum(axis=0))


def sup_grad(S, labels, label_mask):
    grad = np.zeros_like(S)
    if not np.any(label_mask): return grad
    for lab in np.unique(labels[label_mask]):
        mask = (labels == lab) & label_mask
        if mask.sum() == 0: continue
        grad[mask] = S[mask] - np.mean(S[mask], axis=0, keepdims=True)
    return grad


def semi_grad(S, n, label_mask, aw):
    grad = np.zeros_like(S)
    for i in range(n):
        if (not label_mask[i]) and (i in aw):
            grad[i] = S[i] - sum(w * S[j] for j, w in aw[i].items())
    return grad


print('FUSE primitives defined.')

FUSE primitives defined.


## 4. Four Strategy Implementations

In [11]:
def fuse_s1(G, labels, label_mask, X, k=150, eta=0.01,
             ls=1.0, lm=2.0, iters=200, seed=None, nw=10, wl=5, wll=3):
    """Strategy 1: Feature-Initialized Embeddings."""
    if seed: np.random.seed(seed); random.seed(seed)
    A = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    deg = np.array(A.sum(axis=1)).flatten()
    m = max(G.number_of_edges(), 1)
    n = A.shape[0]

    d = X.shape[1]
    if d >= k:
        S = TruncatedSVD(n_components=k, random_state=seed).fit_transform(X)
    else:
        S = np.zeros((n, k)); S[:, :d] = X
    S, _ = np.linalg.qr(S)

    aw = attn_base(S, perform_labeled_random_walks(G, label_mask, labels, nw, wl, wll))
    for _ in range(iters):
        S += eta * (mod_grad(A,S,deg,m) - ls*sup_grad(S,labels,label_mask) - lm*semi_grad(S,n,label_mask,aw))
        S, _ = np.linalg.qr(S)
    return S


def fuse_s2(G, labels, label_mask, X, k=150, eta=0.01,
             ls=1.0, lm=2.0, iters=200, alpha=0.5, seed=None, nw=10, wl=5, wll=3):
    """Strategy 2: Feature-Aware Attention."""
    if seed: np.random.seed(seed); random.seed(seed)
    A = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    deg = np.array(A.sum(axis=1)).flatten()
    m = max(G.number_of_edges(), 1)
    n = A.shape[0]
    S = np.random.randn(n, k); S, _ = np.linalg.qr(S)

    aw = attn_feature_aware(S,
             perform_labeled_random_walks(G, label_mask, labels, nw, wl, wll),
             np.array(X, dtype=float), alpha=alpha)
    for _ in range(iters):
        S += eta * (mod_grad(A,S,deg,m) - ls*sup_grad(S,labels,label_mask) - lm*semi_grad(S,n,label_mask,aw))
        S, _ = np.linalg.qr(S)
    return S


def fuse_s3(G, labels, label_mask, X, k=150, eta=0.01,
             ls=1.0, lm=2.0, lf=0.5, iters=200, seed=None, nw=10, wl=5, wll=3):
    """Strategy 3: Feature Reconstruction Gradient."""
    if seed: np.random.seed(seed); random.seed(seed)
    A = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    deg = np.array(A.sum(axis=1)).flatten()
    m = max(G.number_of_edges(), 1)
    n = A.shape[0]
    S = np.random.randn(n, k); S, _ = np.linalg.qr(S)
    X_arr = np.array(X, dtype=float)

    aw = attn_base(S, perform_labeled_random_walks(G, label_mask, labels, nw, wl, wll))
    for _ in range(iters):
        W, _, _, _ = np.linalg.lstsq(S, X_arr, rcond=None)
        g_feat = (S @ W - X_arr) @ W.T
        S += eta * (mod_grad(A,S,deg,m)
                    - ls*sup_grad(S,labels,label_mask)
                    - lm*semi_grad(S,n,label_mask,aw)
                    - lf*g_feat)
        S, _ = np.linalg.qr(S)
    return S


def build_hybrid_adj(A, X, beta=0.3, knn_k=10):
    feat_sim = cosine_similarity(np.array(X, dtype=float))
    np.fill_diagonal(feat_sim, 0)
    thresh = np.partition(feat_sim, -knn_k, axis=1)[:, -knn_k]
    feat_adj = csr_matrix((feat_sim >= thresh[:, None]).astype(np.float32))
    feat_adj.setdiag(0); feat_adj.eliminate_zeros()
    return (1.0 - beta) * A + beta * feat_adj


def fuse_s4(G, labels, label_mask, X, k=150, eta=0.01,
             ls=1.0, lm=2.0, iters=200, beta=0.3, knn_k=10, seed=None, nw=10, wl=5, wll=3):
    """Strategy 4: Feature-Augmented Adjacency."""
    if seed: np.random.seed(seed); random.seed(seed)
    A_struct = csr_matrix(nx.to_scipy_sparse_array(G, format='csr'))
    A = build_hybrid_adj(A_struct, X, beta=beta, knn_k=knn_k)
    G_h = nx.from_scipy_sparse_array(A)
    deg = np.array(A.sum(axis=1)).flatten()
    m = max(A.nnz // 2, 1)
    n = A.shape[0]
    S = np.random.randn(n, k); S, _ = np.linalg.qr(S)

    aw = attn_base(S, perform_labeled_random_walks(G_h, label_mask, labels, nw, wl, wll))
    for _ in range(iters):
        S += eta * (mod_grad(A,S,deg,m) - ls*sup_grad(S,labels,label_mask) - lm*semi_grad(S,n,label_mask,aw))
        S, _ = np.linalg.qr(S)
    return S


STRATEGY_FNS = {
    's1': fuse_s1,
    's2': fuse_s2,
    's3': fuse_s3,
    's4': fuse_s4,
}

print('All 4 strategy functions defined.')

All 4 strategy functions defined.


## 5. Classifier Infrastructure

In [13]:
class NoMaskGCNConv(GCNConv):
    def compute_mask(self, inputs, mask=None): return None
    def call(self, inputs, training=None, mask=None): return super().call(inputs, mask=None)

class NoMaskGATConv(GATConv):
    def compute_mask(self, inputs, mask=None): return None
    def call(self, inputs, training=None, mask=None): return super().call(inputs, mask=None)

class GCN(tf.keras.Model):
    def __init__(self, nc, seed=42):
        super().__init__()
        I = tf.keras.initializers.GlorotUniform(seed=seed)
        self.c1 = NoMaskGCNConv(16, activation='relu', kernel_initializer=I)
        self.c2 = NoMaskGCNConv(nc, activation='softmax', kernel_initializer=I)
    def call(self, inp, training=False):
        x, a = inp; h = self.c1([x, a]); return self.c2([h, a]), h

class GAT(tf.keras.Model):
    def __init__(self, nc, seed=42):
        super().__init__()
        I = tf.keras.initializers.GlorotUniform(seed=seed)
        self.c1 = NoMaskGATConv(16, attn_heads=8, concat_heads=True, activation='elu', kernel_initializer=I)
        self.c2 = NoMaskGATConv(nc, attn_heads=1, concat_heads=False, activation='softmax', kernel_initializer=I)
    def call(self, inp, training=False):
        x, a = inp; h = self.c1([x, a]); return self.c2([h, a]), h

class GraphSAGE(tf.keras.Model):
    def __init__(self, nc, seed=42):
        super().__init__()
        I = tf.keras.initializers.GlorotUniform(seed=seed)
        self.c1 = GraphSageConv(16, activation='relu', kernel_initializer=I)
        self.c2 = GraphSageConv(nc, activation='softmax', kernel_initializer=I)
    def call(self, inp, training=False):
        x, a = inp; h = self.c1([x, a]); return self.c2([h, a]), h


def to_tf_sparse(A):
    A = A.tocoo()
    return tf.sparse.SparseTensor(
        indices=np.column_stack((A.row, A.col)),
        values=A.data, dense_shape=A.shape)


def train_and_eval(E, A, y_onehot, labels_int, label_mask,
                   clf_name='gcn', epochs=200, seed=42):
    tf.random.set_seed(seed); np.random.seed(seed)
    nc = y_onehot.shape[1]
    tr = np.where(label_mask)[0]
    X_tr = tf.convert_to_tensor(E[tr].astype(np.float32))
    y_tr = y_onehot[tr]
    A_tr = to_tf_sparse(A[tr, :][:, tr])

    model = {'gcn': GCN, 'gat': GAT, 'graphsage': GraphSAGE}[clf_name.lower()](nc, seed=seed)
    opt = tf.keras.optimizers.Adam(1e-2)
    loss_fn = tf.keras.losses.CategoricalCrossentropy()

    t0 = time.time()
    for _ in range(epochs):
        with tf.GradientTape() as tape:
            p, _ = model([X_tr, A_tr], training=True)
            loss = loss_fn(y_tr, p)
        opt.apply_gradients(zip(tape.gradient(loss, model.trainable_variables), model.trainable_variables))
    train_time = time.time() - t0

    p_all, _ = model([tf.convert_to_tensor(E.astype(np.float32)), to_tf_sparse(A)], training=False)
    pred = tf.argmax(p_all, axis=1).numpy()
    mi = np.where(~label_mask)[0]
    return {
        'accuracy':    float(accuracy_score(labels_int[mi], pred[mi])),
        'f1_score':    float(f1_score(labels_int[mi], pred[mi], average='macro')),
        'train_time':  float(train_time),
    }

print('Classifier infrastructure defined.')

Classifier infrastructure defined.


## 6. Run Full Benchmark

In [15]:
def run_benchmark(cfg):
    all_records = []
    strategy_records = {s: [] for s in cfg['strategies']}

    tasks = [(ds, mf, seed)
             for ds   in cfg['datasets']
             for mf   in cfg['mask_fracs']
             for seed in cfg['seeds']]

    for ds_name, mf, seed in tqdm(tasks, desc='Experiments'):
        # Load dataset once per (ds, mf, seed)
        ds = load_dataset(ds_name, root=cfg['data_root'])
        _, label_mask, _ = create_label_mask(ds['labels'], mask_frac=mf, seed=seed)

        known_pct  = int(mf * 100)
        masked_pct = 100 - known_pct

        for strat in cfg['strategies']:
            fn = STRATEGY_FNS[strat]

            # Build strategy-specific kwargs
            common = dict(
                G=ds['G'], labels=ds['labels'], label_mask=label_mask, X=ds['x'],
                k=cfg['emb_dim'], iters=cfg['iterations'], seed=seed,
                nw=cfg['num_walks'], wl=cfg['walk_length'], wll=cfg['walk_length_labelled']
            )
            extra = {}
            if strat == 's2': extra['alpha'] = cfg['s2_alpha']
            if strat == 's3': extra['lf']    = cfg['s3_lambda_feature']
            if strat == 's4': extra.update(beta=cfg['s4_beta'], knn_k=cfg['s4_knn_k'])

            t0 = time.time()
            E  = fn(**common, **extra)
            emb_time = time.time() - t0

            # Save embedding
            emb_dir = os.path.join(SAVE_ROOT, ds_name, f'{known_pct}known_{masked_pct}masked', strat)
            os.makedirs(emb_dir, exist_ok=True)
            with open(os.path.join(emb_dir, f'emb_seed{seed}.pkl'), 'wb') as f:
                pickle.dump(E, f)

            for clf in cfg['classifiers']:
                res = train_and_eval(E, ds['a'], ds['y'], ds['labels'], label_mask,
                                     clf_name=clf, epochs=cfg['clf_epochs'], seed=seed)
                record = dict(
                    dataset=ds_name, seed=seed, mask_frac=mf,
                    known_pct=known_pct, masked_pct=masked_pct,
                    strategy=strat, classifier=clf,
                    emb_time=emb_time,
                    **res
                )
                if strat == 's2': record['alpha']           = cfg['s2_alpha']
                if strat == 's3': record['lambda_feature']  = cfg['s3_lambda_feature']
                if strat == 's4': record.update(beta=cfg['s4_beta'], knn_k=cfg['s4_knn_k'])

                all_records.append(record)
                strategy_records[strat].append(record)

                if cfg['verbose']:
                    print(f'  [{ds_name}|mf={mf}|seed={seed}|{strat}|{clf}]  '
                          f'acc={res["accuracy"]:.4f}  f1={res["f1_score"]:.4f}')

    return all_records, strategy_records


print('Benchmark runner defined. Starting run...')
print(f"Datasets   : {CONFIG['datasets']}")
print(f"Seeds      : {CONFIG['seeds']}")
print(f"Mask fracs : {CONFIG['mask_fracs']}")
print(f"Strategies : {CONFIG['strategies']}")
print(f"Classifiers: {CONFIG['classifiers']}")
total = (len(CONFIG['datasets']) * len(CONFIG['seeds']) * len(CONFIG['mask_fracs'])
         * len(CONFIG['strategies']) * len(CONFIG['classifiers']))
print(f"\nTotal classifier evaluations: {total}")

Benchmark runner defined. Starting run...
Datasets   : ['cora', 'citeseer', 'pubmed', 'wikics', 'photo']
Seeds      : [42]
Mask fracs : [0.7, 0.3]
Strategies : ['s1', 's2', 's3', 's4']
Classifiers: ['gcn', 'gat', 'graphsage']

Total classifier evaluations: 120


In [16]:
all_records, strategy_records = run_benchmark(CONFIG)
print(f'\nBenchmark complete. {len(all_records)} records collected.')

Experiments:   0%|          | 0/10 [00:00<?, ?it/s]


Benchmark complete. 120 records collected.


## 7. Save Results

In [18]:
df_all = pd.DataFrame(all_records)

# ── Per-strategy CSVs ────────────────────────────────────────────────────────
strategy_labels = {
    's1': 'strategy1_feature_init',
    's2': 'strategy2_feature_attention',
    's3': 'strategy3_feature_reconstruction',
    's4': 'strategy4_augmented_adj',
}
for strat, records in strategy_records.items():
    if not records: continue
    fname = os.path.join(SAVE_ROOT, f'{strategy_labels[strat]}_per_run.csv')
    pd.DataFrame(records).to_csv(fname, index=False)
    print(f'Saved: {fname}')

# ── All runs combined ────────────────────────────────────────────────────────
all_path = os.path.join(SAVE_ROOT, 'all_strategies_per_run.csv')
df_all.to_csv(all_path, index=False)
print(f'Saved: {all_path}')

# ── Aggregated: mean ± std across seeds ─────────────────────────────────────
agg = df_all.groupby(['dataset', 'mask_frac', 'strategy', 'classifier']).agg(
    avg_accuracy  =('accuracy',  'mean'),
    std_accuracy  =('accuracy',  'std'),
    avg_f1        =('f1_score',  'mean'),
    std_f1        =('f1_score',  'std'),
    avg_emb_time  =('emb_time',  'mean'),
    avg_train_time=('train_time','mean'),
    n_runs        =('accuracy',  'count'),
).reset_index()

agg['accuracy_pm'] = agg.apply(
    lambda r: f"{r['avg_accuracy']:.4f} ± {r['std_accuracy']:.4f}", axis=1)
agg['f1_pm'] = agg.apply(
    lambda r: f"{r['avg_f1']:.4f} ± {r['std_f1']:.4f}", axis=1)

agg_path = os.path.join(SAVE_ROOT, 'aggregated_results.csv')
agg.to_csv(agg_path, index=False)
print(f'Saved: {agg_path}')

Saved: ./feature rich fuse\benchmark\strategy1_feature_init_per_run.csv
Saved: ./feature rich fuse\benchmark\strategy2_feature_attention_per_run.csv
Saved: ./feature rich fuse\benchmark\strategy3_feature_reconstruction_per_run.csv
Saved: ./feature rich fuse\benchmark\strategy4_augmented_adj_per_run.csv
Saved: ./feature rich fuse\benchmark\all_strategies_per_run.csv
Saved: ./feature rich fuse\benchmark\aggregated_results.csv


## 8. Summary Table

In [20]:
print('=== Aggregated Results (mean ± std across seeds) ===')
pivot = agg[['dataset', 'mask_frac', 'strategy', 'classifier', 'accuracy_pm', 'f1_pm']]
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 30)
print(pivot.to_string(index=False))

=== Aggregated Results (mean ± std across seeds) ===
 dataset  mask_frac strategy classifier  accuracy_pm        f1_pm
citeseer        0.3       s1        gat 0.6088 ± nan 0.5682 ± nan
citeseer        0.3       s1        gcn 0.5852 ± nan 0.5500 ± nan
citeseer        0.3       s1  graphsage 0.6239 ± nan 0.5947 ± nan
citeseer        0.3       s2        gat 0.6256 ± nan 0.5825 ± nan
citeseer        0.3       s2        gcn 0.6050 ± nan 0.5624 ± nan
citeseer        0.3       s2  graphsage 0.6106 ± nan 0.5759 ± nan
citeseer        0.3       s3        gat 0.6685 ± nan 0.6299 ± nan
citeseer        0.3       s3        gcn 0.5208 ± nan 0.4900 ± nan
citeseer        0.3       s3  graphsage 0.5766 ± nan 0.5454 ± nan
citeseer        0.3       s4        gat 0.6325 ± nan 0.5939 ± nan
citeseer        0.3       s4        gcn 0.5603 ± nan 0.5224 ± nan
citeseer        0.3       s4  graphsage 0.5277 ± nan 0.4742 ± nan
citeseer        0.7       s1        gat 0.7154 ± nan 0.6726 ± nan
citeseer        0.7    

In [21]:
# Best strategy per dataset × mask_frac × classifier
print('=== Best Strategy per (dataset, mask_frac, classifier) by Accuracy ===')
best = (agg.sort_values('avg_accuracy', ascending=False)
           .groupby(['dataset', 'mask_frac', 'classifier'])
           .first()
           .reset_index()
           [['dataset', 'mask_frac', 'classifier', 'strategy', 'accuracy_pm', 'f1_pm']])
print(best.to_string(index=False))

best_path = os.path.join(SAVE_ROOT, 'best_strategy_per_setting.csv')
best.to_csv(best_path, index=False)
print(f'\nSaved: {best_path}')

=== Best Strategy per (dataset, mask_frac, classifier) by Accuracy ===
 dataset  mask_frac classifier strategy  accuracy_pm        f1_pm
citeseer        0.3        gat       s3 0.6685 ± nan 0.6299 ± nan
citeseer        0.3        gcn       s2 0.6050 ± nan 0.5624 ± nan
citeseer        0.3  graphsage       s1 0.6239 ± nan 0.5947 ± nan
citeseer        0.7        gat       s3 0.7335 ± nan 0.7016 ± nan
citeseer        0.7        gcn       s2 0.6713 ± nan 0.6310 ± nan
citeseer        0.7  graphsage       s1 0.7214 ± nan 0.6821 ± nan
    cora        0.3        gat       s4 0.8143 ± nan 0.7936 ± nan
    cora        0.3        gcn       s2 0.7769 ± nan 0.7687 ± nan
    cora        0.3  graphsage       s2 0.8001 ± nan 0.7849 ± nan
    cora        0.7        gat       s4 0.8732 ± nan 0.8598 ± nan
    cora        0.7        gcn       s2 0.8214 ± nan 0.8031 ± nan
    cora        0.7  graphsage       s1 0.8571 ± nan 0.8391 ± nan
   photo        0.3        gat       s4 0.9147 ± nan 0.8914 ± nan
   ph

In [22]:
print(f'\n=== All output files in {os.path.abspath(SAVE_ROOT)} ===')
for root, dirs, files in os.walk(SAVE_ROOT):
    level = root.replace(SAVE_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')


=== All output files in C:\Users\user\Downloads\FUSE\modularity_intergrated_version\modularity_intergrated_version\feature rich fuse\benchmark ===
benchmark/
  aggregated_results.csv
  all_strategies_per_run.csv
  best_strategy_per_setting.csv
  strategy1_feature_init_per_run.csv
  strategy2_feature_attention_per_run.csv
  strategy3_feature_reconstruction_per_run.csv
  strategy4_augmented_adj_per_run.csv
  citeseer/
    30known_70masked/
      s1/
        emb_seed42.pkl
      s2/
        emb_seed42.pkl
      s3/
        emb_seed42.pkl
      s4/
        emb_seed42.pkl
    70known_30masked/
      s1/
        emb_seed42.pkl
      s2/
        emb_seed42.pkl
      s3/
        emb_seed42.pkl
      s4/
        emb_seed42.pkl
  cora/
    30known_70masked/
      s1/
        emb_seed42.pkl
      s2/
        emb_seed42.pkl
      s3/
        emb_seed42.pkl
      s4/
        emb_seed42.pkl
    70known_30masked/
      s1/
        emb_seed42.pkl
      s2/
        emb_seed42.pkl
      s3/
        emb